# MedNorm-VI E4 Clean Training

The single current E4 notebook. The implementation that produced the collapsed
run audited in 0043/0044 was removed, not patched; this replaces both the old
full-training notebook and the tiny-overfit diagnostic.

Intended environment: Google Colab, T4 (any CUDA GPU is acceptable; CPU is
refused), Google Drive mounted. Committed with **every run flag False**.

## What went wrong, and what changed

Audit 0044 proved `ALL_BACKGROUND_LOSS_COLLAPSE` by direct measurement: the
final checkpoint predicted `NONE` at all 1,491,764 validation grid cells, and its
decision margin varied by 0.562 nats across the entire split — an
input-independent predictor. Four measured causes, each corrected here:

| measured failure | correction |
| --- | --- |
| per-example loss mean — a 5-word doc outweighed a 162-word doc per cell | batch-global valid-cell reduction, one division per effective batch |
| 577:1 background, no compensation | three candidate recipes, ablated in Stage 2 |
| unshuffled source-grouped order — 10,027 zero-entity examples opened every epoch | deterministic shuffle + stratified source interleaving; positive-aware sampling caps the zero-entity streak at 4 |
| constant LR 2e-5 for all 50,748 steps | backbone 5e-6 / head 1e-3, linear warmup + decay |

## Four gated stages

    Stage 1  dependency and environment preflight
    Stage 2  tiny-overfit recipe ablation      I_AUTHORIZE_E4_TINY_RECIPE_ABLATION
    Stage 3  representative-subset smoke       I_AUTHORIZE_E4_SUBSET_SMOKE
    Stage 4  full T4 training                  I_AUTHORIZE_E4_FULL_TRAINING

The chain **fails closed**. Stage 4 reads the Stage-2 and Stage-3 gate artifacts,
requires both to have passed, requires them to agree on the recipe, and re-checks
the config, code and corpus hashes recorded in them. Flipping the full-training
flag satisfies exactly one of those conditions.

## Supervision scope

The output schema keeps all five organizer types and all seven grid labels. The
governed E4 corpus contains **zero** TEST_NAME and TEST_RESULT mentions, so those
two classifier outputs have no training signal. No synthetic laboratory mention is
generated, and no stage is failed for not predicting them — laboratory extraction
belongs to E1/E2 and the L4 resolver.

## Never

No `internal_test`. No organizer inference. No `output.zip`. No resume from any
superseded checkpoint. No tracked checkpoints.


In [ ]:
# =============================================================================
# STAGE 1a - DEPENDENCY INSTALL. Runs BEFORE any project import.
#
# The prior tiny diagnostic died in Colab on a missing py_vncorenlp. This is an
# executable install, pinned, not a documented prerequisite.
# =============================================================================
%pip install -q py_vncorenlp==0.1.4


In [ ]:
# =============================================================================
# STAGE 1b - ENVIRONMENT PREFLIGHT. Fails loudly and early.
# =============================================================================
from pathlib import Path
import json
import os
import subprocess
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = os.environ.get("MEDNORM_REPO_URL", "https://github.com/vquclinh/MedNorm-VI")
REPO_REF = os.environ.get("MEDNORM_REPO_REF", "main")
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"
RUNTIME_ROOT = Path("/content/mednorm_vi_runtime")

PINNED_MODEL_REVISION = "1c7880f20db59c0054c6de5afd71b012369f6ee4"

# ---------------------------------------------------------------------------
# OPERATOR SETTINGS - committed in the fully disabled state.
# A fresh Run all performs the preflight and stops. It trains nothing.
# ---------------------------------------------------------------------------
RUN_STAGE2_TINY_ABLATION = False
RUN_STAGE3_SUBSET_SMOKE = False
RUN_STAGE4_FULL_TRAINING = False

CONFIRM_STAGE2 = ""
CONFIRM_STAGE3 = ""
CONFIRM_STAGE4 = ""

SEED = 20260728
GRADIENT_ACCUMULATION_STEPS = 8
FULL_EPOCHS = 12

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except Exception as error:  # noqa: BLE001 - Colab-only import
    print(json.dumps({"stage": "drive_mount_skipped", "detail": str(error)}))

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True).stdout.strip()

preflight = {"stage": "preflight", "git_commit": GIT_COMMIT}
failures = []

# py_vncorenlp - the dependency that broke the previous run.
try:
    import py_vncorenlp
    preflight["py_vncorenlp_version"] = getattr(py_vncorenlp, "__version__", "0.1.4")
    preflight["py_vncorenlp_location"] = str(Path(py_vncorenlp.__file__).parent)
except Exception as error:  # noqa: BLE001 - reported, not hidden
    failures.append(f"py_vncorenlp import failed: {error}")

# VnCoreNLP models: download, or resolve an existing cache.
VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
try:
    jar = VNCORENLP_DIR / "VnCoreNLP-1.2.jar"
    if jar.is_file():
        preflight["vncorenlp"] = "resolved_from_cache"
    else:
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
        preflight["vncorenlp"] = "downloaded"
    preflight["vncorenlp_dir"] = str(VNCORENLP_DIR)
    preflight["vncorenlp_jar_present"] = jar.is_file()
except Exception as error:  # noqa: BLE001 - reported, not hidden
    failures.append(f"VnCoreNLP model resolution failed: {error}")

try:
    import torch
    preflight["torch_version"] = torch.__version__
    preflight["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        preflight["gpu_name"] = torch.cuda.get_device_name(0)
        preflight["bf16_supported"] = bool(
            getattr(torch.cuda, "is_bf16_supported", lambda: False)())
        capability = torch.cuda.get_device_capability(0)
        preflight["compute_capability"] = f"{capability[0]}.{capability[1]}"
        # T4 is compute capability 7.5 and has no bf16, so it resolves to fp16 +
        # GradScaler. The runtime capability decides, never the device name.
        preflight["t4_compatible_cuda_runtime"] = capability[0] >= 7
        if not preflight["t4_compatible_cuda_runtime"]:
            failures.append(f"compute capability {capability} is below the T4 baseline")
    else:
        failures.append("CUDA is not available; E4 training refuses the CPU path")
except Exception as error:  # noqa: BLE001 - reported, not hidden
    failures.append(f"torch import failed: {error}")

try:
    import transformers
    preflight["transformers_version"] = transformers.__version__
except Exception as error:  # noqa: BLE001 - reported, not hidden
    failures.append(f"transformers import failed: {error}")

preflight["pinned_phobert_revision"] = PINNED_MODEL_REVISION
preflight["internal_test_prohibited"] = True
preflight["internal_test_accessed"] = False
preflight["failures"] = failures
preflight["ok"] = not failures
print(json.dumps(preflight, indent=2, sort_keys=True))
if failures:
    raise SystemExit("preflight failed; fix the reported dependencies before continuing")


In [ ]:
# =============================================================================
# Project imports. One current E4 package.
# =============================================================================
import hashlib
import time

from mednorm_vi.mention_factory.w2ner import (
    EntitySpan,
    W2NERLabelVocab,
    build_relation_grid_head,
    decode_w2ner_grid,
)
from mednorm_vi.training.phase2.common import canonical_json_sha256, sha256_file
from mednorm_vi.training.phase2.e4 import (
    STAGE2_RECIPE_NAMES,
    BatchGlobalAccumulator,
    BestFinalRecord,
    EpochTelemetry,
    ExampleIndex,
    GateArtifact,
    RecipeResult,
    ReproductionCheck,
    SubsetResult,
    TinyEpochSignal,
    TinyOverfitStopPolicy,
    ValidationSnapshot,
    assert_full_training_allowed,
    assert_gate_uses_best_metrics,
    assert_stage_authorized,
    build_epoch_order,
    build_recipe,
    evaluate_collapse_guard,
    measure_order,
    plan_gradient_accumulation,
    plan_schedule,
    reject_superseded_checkpoint,
    resolve_mixed_precision_policy,
    select_recipe,
    stage2_recipes,
)
from mednorm_vi.training.phase2.e4.alignment import (
    atomic_relation_head_input_dim,
    build_atomic_projection,
    build_w2ner_batch_contract_from_segmented_words,
    decode_w2ner_logits,
    prepare_phobert_word_inputs,
    project_to_atomic_word_embeddings,
)
from mednorm_vi.training.phase2.e4.contracts import (
    E4_GOVERNED_TRAIN_SHA256,
    E4_GOVERNED_VALIDATION_SHA256,
    E4_MODEL_ID,
    E4_SUPERVISED_TYPES,
    assert_weight_format_loadable,
    build_e4_history_row,
    build_e4_resolved_config,
    e4_checkpoint_payload,
    resolve_governed_split_by_sha256,
    resolve_phobert_weight_format,
    validate_phobert_encoder_load_report,
)
from mednorm_vi.training.phase2.e4.gates import (
    SUBSET_GATE_FILENAME,
    TINY_GATE_FILENAME,
    TINY_TARGET_EXACT_F1,
    assert_real_reproduction,
    render_recipe_table,
    required_supervised_types,
)
from mednorm_vi.training.phase2.e4.training import (
    BEST_STATE_POST_RELOAD,
    BEST_STATE_PRE_SERIALIZATION,
    POSITIVE_CLASS_ORDER,
    BestCheckpointSelector,
    assert_not_collapsed_when_marking_trained,
    assert_training_device,
    build_training_accounting,
)
from mednorm_vi.training.phobert_alignment import (
    map_segmented_words,
    resolve_segmented_text,
    segmented_text_to_words,
)

# One current artifact directory:
#   /content/drive/MyDrive/MedNorm-VI/artifacts/e4_current
# Recipe candidates and non-best subset checkpoints are pruned after selection,
# and nothing under it is ever tracked by Git.
CURRENT_DIR = DRIVE_ROOT / "artifacts" / "e4_current"
GATE_DIR = CURRENT_DIR / "gates"
TINY_DIR = CURRENT_DIR / "stage2_tiny"
SUBSET_DIR = CURRENT_DIR / "stage3_subset"
for directory in (CURRENT_DIR, GATE_DIR, TINY_DIR, SUBSET_DIR,
                  CURRENT_DIR / "checkpoints", CURRENT_DIR / "logs", RUNTIME_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)

VOCAB = W2NERLabelVocab()
MAX_WORDS = 256
MAX_MODEL_TOKENS = 256
WEIGHT_FORMAT = resolve_phobert_weight_format(E4_MODEL_ID, PINNED_MODEL_REVISION)
assert_weight_format_loadable(WEIGHT_FORMAT)

# The code hash binds the gate artifacts to this exact implementation.
CODE_SHA256 = hashlib.sha256(b"".join(
    sorted(p.read_bytes() for p in
           (REPO_DIR / "src" / "mednorm_vi" / "training" / "phase2" / "e4").glob("*.py"))
)).hexdigest()
print(json.dumps({"stage": "e4_code_identity", "code_sha256": CODE_SHA256,
                  "supervised_types": list(E4_SUPERVISED_TYPES),
                  "stage2_recipes": list(STAGE2_RECIPE_NAMES)}, indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# Governed corpus. Resolved by authoritative SHA-256, never by filename.
# internal_test is never resolved, opened or copied.
# =============================================================================
import py_vncorenlp

SEARCH_ROOTS = (DRIVE_ROOT / "data", DRIVE_ROOT / "data" / "derived", REPO_DIR / "data")
TRAIN_RESOLUTION = resolve_governed_split_by_sha256(
    split="train", expected_sha256=E4_GOVERNED_TRAIN_SHA256, search_roots=SEARCH_ROOTS)
VALIDATION_RESOLUTION = resolve_governed_split_by_sha256(
    split="validation", expected_sha256=E4_GOVERNED_VALIDATION_SHA256,
    search_roots=SEARCH_ROOTS)
CORPUS_SHA256 = {"train": TRAIN_RESOLUTION.sha256,
                 "validation": VALIDATION_RESOLUTION.sha256}

def load_rows(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for row_index, line in enumerate(handle):
            if not line.strip():
                continue
            payload = json.loads(line)
            entities = tuple(
                EntitySpan(int(e["start"]), int(e["end"]),
                           str(e.get("target_type") or e.get("type")), str(e["text"]))
                for e in payload.get("entities", []))
            rows.append({
                "row_index": row_index,
                "document_id": str(payload.get("document_id", "")),
                "source_dataset": str(payload.get("source_dataset", "")),
                "text": str(payload["text"]),
                "entities": entities,
            })
    return rows

TRAIN_ROWS = load_rows(TRAIN_RESOLUTION.path)
VALIDATION_ROWS = load_rows(VALIDATION_RESOLUTION.path)
print(json.dumps({
    "stage": "corpus_resolved",
    "train_examples": len(TRAIN_ROWS),
    "validation_examples": len(VALIDATION_ROWS),
    "corpus_sha256": CORPUS_SHA256,
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))

_previous_cwd = os.getcwd()
os.chdir(VNCORENLP_DIR)
SEGMENTER = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
os.chdir(_previous_cwd)

def segment(text: str) -> str:
    return " ".join(SEGMENTER.word_segment(text))

def build_contract(row):
    segmented_text, _source = resolve_segmented_text(row["text"], segment)
    model_words = map_segmented_words(row["text"], segmented_text_to_words(segmented_text))
    return build_w2ner_batch_contract_from_segmented_words(
        row["document_id"], row["text"], row["entities"], model_words,
        max_words=MAX_WORDS, vocab=VOCAB)


In [ ]:
# =============================================================================
# Shared training engine. Every stage runs the SAME code path, so a recipe that
# passes Stage 2 is the recipe Stage 3 and Stage 4 execute.
# =============================================================================
import math

import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert_training_device(DEVICE.type)
BF16 = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
PRECISION = resolve_mixed_precision_policy("fp16", device_type=DEVICE.type, bf16_supported=BF16)

TOKENIZER = AutoTokenizer.from_pretrained(
    E4_MODEL_ID, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR),
    use_fast=False)

def fresh_model():
    """Pinned pretrained backbone + a NEWLY initialized relation head."""
    base, info = AutoModel.from_pretrained(
        E4_MODEL_ID, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR),
        use_safetensors=WEIGHT_FORMAT.use_safetensors, output_loading_info=True)
    validate_phobert_encoder_load_report(
        missing_keys=tuple(info.get("missing_keys", ())),
        unexpected_keys=tuple(info.get("unexpected_keys", ())))
    head = build_relation_grid_head(
        atomic_relation_head_input_dim(base.config.hidden_size), len(VOCAB.labels))
    return base.to(DEVICE), head.to(DEVICE)

def build_optimizer(base, head, recipe):
    """Differential groups. The fresh head must not inherit the backbone's LR."""
    groups = [
        {"params": list(base.parameters()), "lr": recipe.optimizer.backbone_lr,
         "name": "backbone"},
        {"params": list(head.parameters()), "lr": recipe.optimizer.head_lr,
         "name": "relation_head"},
    ]
    return torch.optim.AdamW(groups, weight_decay=recipe.optimizer.weight_decay)

def word_embeddings(base, contract):
    encoding = prepare_phobert_word_inputs(
        TOKENIZER, contract.segmented_words, max_length=MAX_MODEL_TOKENS)
    projection = build_atomic_projection(
        contract.grid.original_text, contract.segmented_words, encoding,
        atomic_words=contract.atomic_words)
    inputs = {k: torch.tensor([v], dtype=torch.long, device=DEVICE)
              for k, v in encoding.model_inputs.items()}
    outputs = base(**inputs)
    return project_to_atomic_word_embeddings(outputs.last_hidden_state[0], projection)

def cell_losses(logits, labels, recipe):
    """Per-cell CE plus the mask of positive cells.

    For `hard_negative_ce` the easy negatives are DROPPED here, deterministically
    by (-loss, flat index): every positive cell survives, and only the top
    `ratio x positives` background cells do. No positive cell is ever discarded.
    """
    flat_logits = logits.reshape(-1, len(VOCAB.labels))
    flat_labels = labels.reshape(-1)
    per_cell = nn.functional.cross_entropy(flat_logits, flat_labels, reduction="none")
    positive = flat_labels != VOCAB.none_id
    selected_negatives = None
    total_negatives = int((~positive).sum())
    if recipe.objective == "hard_negative":
        keep = recipe.hard_negative.keep_count(
            positive_cells=int(positive.sum()), background_cells=total_negatives)
        background_index = (~positive).nonzero(as_tuple=True)[0]
        background_loss = per_cell[background_index]
        # Deterministic: sort by loss descending, ties broken by flat index.
        order = torch.argsort(background_loss, descending=True, stable=True)
        chosen = background_index[order[:keep]]
        mask = positive.clone()
        mask[chosen] = True
        per_cell = per_cell[mask]
        positive = positive[mask]
        selected_negatives = int(keep)
    return per_cell, positive, selected_negatives, total_negatives

def evaluate(base, head, rows, contracts):
    """Forward-only exact-span evaluation plus the cell-level collapse signals."""
    base.eval()
    head.eval()
    predicted_total = gold_total = true_positives = 0
    nnw = thw = 0
    thw_by_type = dict.fromkeys(VOCAB.type_order, 0)
    gold_positive = gold_positive_background = 0
    positive_correct = 0
    by_type = dict.fromkeys(VOCAB.type_order, 0)
    per_class_total = dict.fromkeys(POSITIVE_CLASS_ORDER, 0)
    per_class_correct = dict.fromkeys(POSITIVE_CLASS_ORDER, 0)
    none_logit_sum = 0.0
    margin_sum = 0.0
    with torch.no_grad():
        for contract in contracts:
            embeddings = word_embeddings(base, contract)
            mask = torch.tensor([contract.grid.pair_mask], dtype=torch.bool, device=DEVICE)
            logits = head(embeddings, mask)[0]
            size = len(contract.grid.words)
            window = logits[:size, :size, :]
            argmax = window.argmax(dim=-1).tolist()
            none_logits = window[:, :, VOCAB.none_id].tolist()
            without_none = torch.cat(
                (window[:, :, :VOCAB.none_id], window[:, :, VOCAB.none_id + 1:]),
                dim=-1).max(dim=-1).values.tolist()
            for r in range(size):
                for c in range(size):
                    predicted_label = argmax[r][c]
                    gold_label = contract.grid.labels[r][c]
                    if predicted_label == VOCAB.nnw_id:
                        nnw += 1
                    entity_type = VOCAB.thw_type(predicted_label)
                    if entity_type is not None:
                        thw += 1
                        thw_by_type[entity_type] = thw_by_type.get(entity_type, 0) + 1
                    if gold_label != VOCAB.none_id:
                        gold_positive += 1
                        none_logit_sum += none_logits[r][c]
                        margin_sum += without_none[r][c] - none_logits[r][c]
                        if predicted_label == VOCAB.none_id:
                            gold_positive_background += 1
                        if predicted_label == gold_label:
                            positive_correct += 1
                        gold_name = VOCAB.labels[gold_label]
                        if gold_name in per_class_total:
                            per_class_total[gold_name] += 1
                            if predicted_label == gold_label:
                                per_class_correct[gold_name] += 1
            predicted = set(decode_w2ner_logits(contract, logits.tolist()))
            gold = {(s.start, s.end, s.entity_type) for s in decode_w2ner_grid(contract.grid)}
            predicted_total += len(predicted)
            gold_total += len(gold)
            true_positives += len(predicted & gold)
            for _s, _e, entity_type in predicted:
                by_type[entity_type] = by_type.get(entity_type, 0) + 1
    precision = true_positives / predicted_total if predicted_total else 0.0
    recall = true_positives / gold_total if gold_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "validation_exact_precision": precision,
        "validation_exact_recall": recall,
        "validation_exact_f1": f1,
        "validation_predicted_total": predicted_total,
        "validation_gold_total": gold_total,
        "validation_true_positive": true_positives,
        "validation_nnw_predictions": nnw,
        "validation_thw_predictions": thw,
        "thw_predictions_by_type": thw_by_type,
        "predictions_by_type": by_type,
        "gold_positive_cells": gold_positive,
        "gold_positive_background_rate": (
            gold_positive_background / gold_positive if gold_positive else 1.0),
        "positive_cell_accuracy": (
            positive_correct / gold_positive if gold_positive else 0.0),
        "per_positive_class_accuracy": {
            name: (per_class_correct[name] / per_class_total[name]
                   if per_class_total[name] else 0.0)
            for name in POSITIVE_CLASS_ORDER},
        "per_positive_class_cells": dict(per_class_total),
        "mean_none_logit_on_gold_positive": (
            none_logit_sum / gold_positive if gold_positive else 0.0),
        "strongest_non_none_margin_on_gold_positive": (
            margin_sum / gold_positive if gold_positive else 0.0),
        "internal_test_accessed": False,
    }

def train_recipe(rows, contracts, recipe, *, epochs, accumulation_steps,
                 eval_rows=None, eval_contracts=None, collapse_guard=True,
                 history_path=None, tiny_policy=None):
    """One recipe, one run. Batch-global valid-cell reduction throughout.

    ``tiny_policy`` selects the Stage-2 stopping contract. When it is supplied
    the full-training validation-patience stopper is NOT used: a tiny run ends
    when it meets the gate, exhausts its epoch bound, or breaks numerically.
    Audit 0046 records what happened without it — patience-3 stopped every
    recipe at epoch 4, on 12 of 60 warmup steps.
    """
    base, head = fresh_model()
    optimizer = build_optimizer(base, head, recipe)
    scaler = torch.amp.GradScaler(DEVICE.type) if PRECISION.use_grad_scaler else None
    plan = plan_gradient_accumulation(
        len(rows), accumulation_steps=accumulation_steps, epochs=epochs)
    total_steps = plan.expected_optimizer_steps
    index = [ExampleIndex(r["row_index"], r["source_dataset"], len(r["entities"]))
             for r in rows]
    by_row = {r["row_index"]: (r, c)
              for r, c in zip(rows, contracts, strict=True)}
    selector = BestCheckpointSelector(patience=3)
    best_metrics = None
    best_epoch = 0
    best_state_changed_at = []
    head_grad_norm = backbone_grad_norm = 0.0
    schedule_plan = plan_schedule(
        examples=len(rows), accumulation_steps=accumulation_steps,
        epoch_bound=epochs, warmup_ratio=recipe.schedule.warmup_ratio)
    tiny_signals = []
    stopped_reason = ""
    autocast_dtype = torch.bfloat16 if PRECISION.mode == "bf16" else torch.float16
    snapshots, compositions = [], []
    steps = backwards = examples = 0
    losses = {"total": 0.0, "positive": 0.0, "background": 0.0}
    best_state = None

    for epoch in range(1, epochs + 1):
        order = build_epoch_order(index, data_order=recipe.data_order, seed=SEED, epoch=epoch)
        compositions.append(measure_order(order, epoch=epoch, data_order=recipe.data_order).as_dict())
        base.train()
        head.train()
        epoch_acc = BatchGlobalAccumulator()
        epoch_selected_negatives = 0
        epoch_total_negatives = 0
        optimizer.zero_grad(set_to_none=True)
        pending = []
        for micro_index, item in enumerate(order):
            row, contract = by_row[item.row_index]
            with torch.autocast(device_type=DEVICE.type, dtype=autocast_dtype,
                                enabled=PRECISION.autocast_enabled):
                embeddings = word_embeddings(base, contract)
                mask = torch.tensor([contract.grid.pair_mask], dtype=torch.bool, device=DEVICE)
                labels = torch.tensor([contract.grid.labels], dtype=torch.long, device=DEVICE)
                logits = head(embeddings, mask)
                per_cell, positive, chosen, total_negatives = cell_losses(
                    logits, labels, recipe)
            # The per-cell tensor stays attached: group_balanced_ce needs the
            # positive and background sums separately and differentiably.
            pending.append((per_cell, positive.detach()))
            if chosen is not None:
                epoch_selected_negatives += chosen
                epoch_total_negatives += total_negatives
            backwards += 1
            examples += 1
            if plan.is_optimizer_step_boundary(micro_index):
                # ONE reduction per effective batch, dispatched by the recipe.
                # Numerators and counts are accumulated across every microbatch
                # of the batch and divided once, so accumulation stays exactly
                # equivalent to a single large batch.
                positive_sum = sum(cells[mask].sum() for cells, mask in pending)
                background_sum = sum(cells[~mask].sum() for cells, mask in pending)
                positive_count = sum(int(mask.sum()) for _c, mask in pending)
                background_count = sum(
                    int((~mask).sum()) for _c, mask in pending)
                total_cells = positive_count + background_count

                if recipe.reduction == "batch_global_group_balanced_mean":
                    weights = recipe.group_weights
                    if positive_count and background_count:
                        batch_loss = (
                            weights.positive * positive_sum / positive_count
                            + weights.background * background_sum / background_count)
                    elif positive_count:
                        batch_loss = positive_sum / positive_count
                    else:
                        # Explicit background-only fallback: an effective batch
                        # with no positive cell has no positive mean to combine.
                        batch_loss = background_sum / background_count
                else:
                    # reference_ce: every valid cell. hard_negative_ce: the
                    # positives plus the negatives the objective kept.
                    batch_loss = (positive_sum + background_sum) / total_cells

                if scaler is not None:
                    scaler.scale(batch_loss).backward()
                    scaler.unscale_(optimizer)
                else:
                    batch_loss.backward()
                # Clip each group separately so its true gradient norm is
                # observable; the collapsed run reported neither.
                backbone_grad_norm = float(torch.nn.utils.clip_grad_norm_(
                    optimizer.param_groups[0]["params"],
                    recipe.schedule.max_grad_norm))
                head_grad_norm = float(torch.nn.utils.clip_grad_norm_(
                    optimizer.param_groups[1]["params"],
                    recipe.schedule.max_grad_norm))
                multiplier = recipe.schedule.multiplier_at(steps, total_steps)
                for group in optimizer.param_groups:
                    base_lr = (recipe.optimizer.backbone_lr if group["name"] == "backbone"
                               else recipe.optimizer.head_lr)
                    group["lr"] = base_lr * multiplier
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                steps += 1
                for cells, positive_mask in pending:
                    detached = cells.detach()
                    epoch_acc.observe_microbatch(
                        loss_sum=float(detached.sum()), cells=int(detached.numel()),
                        positive_sum=float(detached[positive_mask].sum()),
                        positive_cells=int(positive_mask.sum()))
                pending = []
        losses = epoch_acc.loss_breakdown()
        metrics = evaluate(base, head, eval_rows or rows, eval_contracts or contracts)
        snapshots.append(ValidationSnapshot(
            epoch=epoch, predicted_mentions=metrics["validation_predicted_total"],
            gold_mentions=metrics["validation_gold_total"],
            true_positives=metrics["validation_true_positive"],
            thw_predictions=metrics["validation_thw_predictions"],
            nnw_predictions=metrics["validation_nnw_predictions"],
            gold_positive_background_rate=metrics["gold_positive_background_rate"],
            train_loss=losses["total"]))
        # BEST state and BEST metrics are captured together, always. Audit 0047:
        # capturing the state here while returning the LAST epoch's metrics made
        # the reproduction check compare two different models.
        best_state_changed = selector.observe(
            epoch=epoch, exact_f1=metrics["validation_exact_f1"])
        if best_state_changed:
            best_state = {"base_model": {k: v.detach().cpu().clone()
                                         for k, v in base.state_dict().items()},
                          "w2ner_head": {k: v.detach().cpu().clone()
                                         for k, v in head.state_dict().items()}}
            best_metrics = dict(metrics)
            best_epoch = epoch
            best_state_changed_at.append(epoch)
        row_record = build_e4_history_row(
            epoch=epoch, mode="train", recipe_name=recipe.name, losses=losses,
            validation_metrics=metrics, optimizer_steps=steps,
            backward_passes=backwards, examples_processed=examples,
            backbone_learning_rate=optimizer.param_groups[0]["lr"],
            head_learning_rate=optimizer.param_groups[1]["lr"])
        print(json.dumps(row_record, sort_keys=True))
        if history_path is not None:
            with Path(history_path).open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(row_record, sort_keys=True) + "\n")
        verdict = evaluate_collapse_guard(snapshots)
        if collapse_guard and verdict.collapsed:
            print(json.dumps({"stage": "collapse_guard_fired", **verdict.as_dict()},
                             indent=2, sort_keys=True))
            stopped_reason = "collapse_guard_fired"
            break

        if tiny_policy is not None:
            # STAGE 2 PATH. No validation patience; the collapse guard is off.
            tiny_signals.append(TinyEpochSignal(
                epoch=epoch, optimizer_steps=steps,
                exact_f1=metrics["validation_exact_f1"],
                predicted_mentions=metrics["validation_predicted_total"],
                positive_cell_accuracy=metrics["positive_cell_accuracy"],
                types_predicted=tuple(
                    t for t, c in metrics["thw_predictions_by_type"].items() if c > 0),
                loss_total=losses["total"],
                loss_is_finite=math.isfinite(losses["total"])))
            if tiny_policy.should_heartbeat(epoch):
                telemetry = EpochTelemetry(
                    epoch=epoch, recipe=recipe.name, optimizer_steps=steps,
                    best_exact_f1=(best_metrics or metrics)["validation_exact_f1"],
                    final_exact_f1=metrics["validation_exact_f1"],
                    positive_cell_accuracy=metrics["positive_cell_accuracy"],
                    per_positive_class_accuracy=metrics["per_positive_class_accuracy"],
                    gold_positive_predicted_as_none_rate=metrics[
                        "gold_positive_background_rate"],
                    loss_positive=losses["positive"],
                    loss_background=losses["background"],
                    positive_cells=int(losses["positive_cells"]),
                    background_cells=int(losses["background_cells"]),
                    head_grad_norm=head_grad_norm,
                    backbone_grad_norm=backbone_grad_norm,
                    mean_none_logit_on_gold_positive=metrics[
                        "mean_none_logit_on_gold_positive"],
                    strongest_non_none_margin_on_gold_positive=metrics[
                        "strongest_non_none_margin_on_gold_positive"],
                    best_epoch=best_epoch, best_state_changed=best_state_changed,
                    selected_hard_negatives=(
                        epoch_selected_negatives
                        if recipe.objective == "hard_negative" else None),
                    total_background_candidates=(
                        epoch_total_negatives
                        if recipe.objective == "hard_negative" else None))
                print(json.dumps({**telemetry.as_dict(),
                                  **schedule_plan.realized(
                                      optimizer_steps=steps, epochs=epoch).as_dict()},
                                 sort_keys=True))
            stop, stopped_reason = tiny_policy.decide(tiny_signals)
            if stop:
                print(json.dumps({"stage": "tiny_stopped", "recipe": recipe.name,
                                  "reason": stopped_reason, "epoch": epoch},
                                 sort_keys=True))
                break
            continue

        # FULL-TRAINING PATH. Validation-patience early stopping is unchanged.
        if selector.should_stop:
            stopped_reason = "early_stopping_patience"
            break

    # best_* and final_* are separate keys. There is no bare "metrics".
    record = BestFinalRecord(
        best_epoch=best_epoch or epoch,
        best_metrics=best_metrics or dict(metrics),
        final_epoch=epoch, final_metrics=dict(metrics),
        best_state_changed_at=tuple(best_state_changed_at))
    return {"best_metrics": record.best_metrics, "best_epoch": record.best_epoch,
            "final_metrics": record.final_metrics, "final_epoch": record.final_epoch,
            "best_final": record, "losses": losses,
            "collapsed": collapse_guard and evaluate_collapse_guard(snapshots).collapsed,
            "verdict": evaluate_collapse_guard(snapshots), "selector": selector,
            "best_state": best_state, "compositions": compositions, "steps": steps,
            "backwards": backwards, "examples": examples, "plan": plan,
            "schedule": schedule_plan.realized(optimizer_steps=steps, epochs=epoch),
            "stopped_reason": stopped_reason, "tiny_signals": tiny_signals}


In [ ]:
# =============================================================================
# STAGE 2 - TINY-OVERFIT RECIPE ABLATION
#
# All three recipes, identical examples / revision / seed / epoch bound /
# evaluation / decoder / precision. Pass requires exact F1 >= 0.95 AND real
# predictions AND every present supervised type. Grid-cell accuracy is never a
# criterion: an all-background model already scores ~0.998 on it.
# =============================================================================
assert_stage_authorized("tiny_recipe_ablation", CONFIRM_STAGE2,
                        enabled=RUN_STAGE2_TINY_ABLATION)

TINY_TARGET = 12
TINY_EPOCH_BOUND = 200

def select_tiny_rows(rows, target=TINY_TARGET):
    """Deterministic: earliest eligible example covering each supervised type."""
    eligible = []
    for row in rows:
        if not row["entities"]:
            continue
        contract = build_contract(row)
        if len(contract.grid.words) > 64:
            continue
        eligible.append(row)
    quota = max(1, target // len(E4_SUPERVISED_TYPES))
    taken, per_type = {}, dict.fromkeys(E4_SUPERVISED_TYPES, 0)
    for entity_type in E4_SUPERVISED_TYPES:
        for row in eligible:
            if len(taken) >= target or per_type[entity_type] >= quota:
                break
            types = {e.entity_type for e in row["entities"]}
            if row["row_index"] in taken or entity_type not in types:
                continue
            taken[row["row_index"]] = row
            for present in types:
                if present in per_type:
                    per_type[present] += 1
    for row in eligible:
        if len(taken) >= target:
            break
        taken.setdefault(row["row_index"], row)
    return [taken[k] for k in sorted(taken)]

TINY_ROWS = select_tiny_rows(TRAIN_ROWS)
TINY_CONTRACTS = [build_contract(row) for row in TINY_ROWS]
TINY_TYPES_PRESENT = sorted({e.entity_type for r in TINY_ROWS for e in r["entities"]})
TINY_REQUIRED = required_supervised_types(TINY_TYPES_PRESENT)
print(json.dumps({
    "stage": "stage2_selection",
    "examples": len(TINY_ROWS),
    "row_indices": [r["row_index"] for r in TINY_ROWS],
    "gold_mentions": sum(len(r["entities"]) for r in TINY_ROWS),
    "types_present": TINY_TYPES_PRESENT,
    "required_types": list(TINY_REQUIRED),
}, indent=2, sort_keys=True))

def evaluate_state(state, rows, contracts):
    """Restore a state dict into a FRESH model and evaluate. No reuse."""
    base, head = fresh_model()
    base_report = base.load_state_dict(state["base_model"], strict=False)
    head_report = head.load_state_dict(state["w2ner_head"], strict=False)
    missing = tuple(sorted(base_report.missing_keys)) + tuple(sorted(head_report.missing_keys))
    unexpected = tuple(sorted(base_report.unexpected_keys)) + tuple(sorted(head_report.unexpected_keys))
    metrics = evaluate(base, head, rows, contracts)
    del base, head
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return metrics, missing, unexpected

def reproduction_metrics(metrics):
    return {
        "exact_precision": metrics["validation_exact_precision"],
        "exact_recall": metrics["validation_exact_recall"],
        "exact_f1": metrics["validation_exact_f1"],
        "predicted_mentions": float(metrics["validation_predicted_total"]),
        "positive_cell_accuracy": metrics["positive_cell_accuracy"],
    }

TINY_POLICY_BASE = dict(
    epoch_bound=TINY_EPOCH_BOUND, target_exact_f1=TINY_TARGET_EXACT_F1,
    required_types=tuple(TINY_REQUIRED), heartbeat_every_n_epochs=5,
    allow_fail_fast=False)

results = []
for recipe in stage2_recipes():
    schedule_plan = plan_schedule(
        examples=len(TINY_ROWS), accumulation_steps=4, epoch_bound=TINY_EPOCH_BOUND,
        warmup_ratio=recipe.schedule.warmup_ratio)
    policy = TinyOverfitStopPolicy(
        warmup_steps=schedule_plan.warmup_steps, **TINY_POLICY_BASE)
    print(json.dumps({"stage": "tiny_recipe_start", "recipe": recipe.name,
                      "policy": policy.as_dict(), "schedule": schedule_plan.as_dict()},
                     indent=2, sort_keys=True))
    started = time.monotonic()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    outcome = train_recipe(TINY_ROWS, TINY_CONTRACTS, recipe, epochs=TINY_EPOCH_BOUND,
                           accumulation_steps=4, collapse_guard=False,
                           tiny_policy=policy)
    elapsed = time.monotonic() - started
    peak = (torch.cuda.max_memory_allocated() / 1024 ** 3) if DEVICE.type == "cuda" else 0.0
    # THE GATE JUDGES THE BEST STATE — the one that gets saved. Judging the final
    # epoch while saving the best state is the Audit-0047 defect.
    metrics = outcome["best_metrics"]
    realized = outcome["schedule"]
    print(json.dumps({"stage": "tiny_best_vs_final", "recipe": recipe.name,
                      **outcome["best_final"].as_dict()}, indent=2, sort_keys=True))

    # -----------------------------------------------------------------------
    # REAL save/reload reproduction: save -> fresh architecture -> restore ->
    # re-evaluate the SAME 12 examples -> compare within tolerance.
    # -----------------------------------------------------------------------
    # 1. Evaluate best_state in a fresh model BEFORE serialization, so both
    #    sides of the comparison are the same state.
    before_metrics, _bm, _bu = evaluate_state(
        outcome["best_state"], TINY_ROWS, TINY_CONTRACTS)
    assert_gate_uses_best_metrics(BEST_STATE_PRE_SERIALIZATION)
    tmp = TINY_DIR / f"{recipe.name}.pt"
    payload = e4_checkpoint_payload(
        mode="tiny", config_sha256=CODE_SHA256, model_revision=PINNED_MODEL_REVISION,
        tokenizer_revision=PINNED_MODEL_REVISION, parameter_count=0,
        recipe_name=recipe.name)
    payload["model_state"] = outcome["best_state"]
    torch.save(payload, tmp)
    digest = sha256_file(tmp)
    reloaded = torch.load(tmp, map_location="cpu", weights_only=False)
    reject_superseded_checkpoint(reloaded)
    # 5. Re-evaluate the SAME best state after reload, in another fresh model.
    after_metrics, missing, unexpected = evaluate_state(
        reloaded["model_state"], TINY_ROWS, TINY_CONTRACTS)
    reproduction = ReproductionCheck(
        checkpoint_sha256=digest,
        metrics_before=reproduction_metrics(before_metrics),
        metrics_after=reproduction_metrics(after_metrics),
        missing_keys=missing, unexpected_keys=unexpected,
        fresh_model_evaluated=True, examples_evaluated=len(TINY_ROWS),
        metrics_before_source=BEST_STATE_PRE_SERIALIZATION,
        metrics_after_source=BEST_STATE_POST_RELOAD,
        eval_mode=True, deterministic_evaluation=True)
    print(json.dumps({"stage": "tiny_save_reload", "recipe": recipe.name,
                      **reproduction.as_dict()}, indent=2, sort_keys=True))
    del reloaded
    tmp.unlink(missing_ok=True)   # no candidate checkpoint is retained

    results.append(RecipeResult(
        recipe=recipe.name,
        exact_precision=metrics["validation_exact_precision"],
        exact_recall=metrics["validation_exact_recall"],
        exact_f1=metrics["validation_exact_f1"],
        predicted_mentions=metrics["validation_predicted_total"],
        gold_mentions=metrics["validation_gold_total"],
        false_positives=metrics["validation_predicted_total"] - metrics["validation_true_positive"],
        positive_cell_accuracy=metrics["positive_cell_accuracy"],
        gold_positive_background_rate=metrics["gold_positive_background_rate"],
        nnw_predictions=metrics["validation_nnw_predictions"],
        thw_predictions_by_type=metrics["thw_predictions_by_type"],
        loss_total=outcome["losses"]["total"],
        loss_positive=outcome["losses"]["positive"],
        loss_background=outcome["losses"]["background"],
        seconds=elapsed, peak_vram_gib=peak, reproduction=reproduction,
        schedule={**realized.as_dict(), **outcome["best_final"].as_dict()},
        stopped_reason=outcome["stopped_reason"]))

SELECTED, TINY_REPORT = select_recipe(results, required_types=TINY_REQUIRED)
print(render_recipe_table(results))
print(json.dumps(TINY_REPORT, indent=2, sort_keys=True))
(TINY_DIR / "recipe_comparison.json").write_text(
    json.dumps(TINY_REPORT, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(TINY_DIR / "recipe_comparison.md").write_text(
    render_recipe_table(results) + "\n", encoding="utf-8")

GateArtifact(
    stage="tiny_recipe_ablation", passed=SELECTED is not None,
    recipe=SELECTED.recipe if SELECTED else "",
    config_sha256=CODE_SHA256, code_sha256=CODE_SHA256,
    corpus_sha256=CORPUS_SHA256, detail=TINY_REPORT,
).write(GATE_DIR / TINY_GATE_FILENAME)

if SELECTED is None:
    raise SystemExit(
        "no recipe passed the tiny-overfit gate; subset and full training must not run")
print(json.dumps({"stage": "stage2_passed", "selected_recipe": SELECTED.recipe},
                 indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 3 - REPRESENTATIVE SUBSET SMOKE. Only the selected recipe.
# =============================================================================
assert_stage_authorized("subset_smoke", CONFIRM_STAGE3, enabled=RUN_STAGE3_SUBSET_SMOKE)

TINY_GATE = json.loads((GATE_DIR / TINY_GATE_FILENAME).read_text(encoding="utf-8"))
if not TINY_GATE["passed"]:
    raise SystemExit("stage 2 did not pass; the subset smoke is refused")
SELECTED_RECIPE = build_recipe(TINY_GATE["recipe"])

SUBSET_PER_SOURCE = 120
SUBSET_ZERO_ENTITY_PER_SOURCE = 40

def build_subset(rows):
    """Deterministic, covers every source and both positive and zero-entity rows."""
    chosen, positive_seen, zero_seen = [], {}, {}
    for row in rows:
        source = row["source_dataset"]
        if row["entities"]:
            if positive_seen.get(source, 0) >= SUBSET_PER_SOURCE:
                continue
            positive_seen[source] = positive_seen.get(source, 0) + 1
        else:
            if zero_seen.get(source, 0) >= SUBSET_ZERO_ENTITY_PER_SOURCE:
                continue
            zero_seen[source] = zero_seen.get(source, 0) + 1
        chosen.append(row)
    return chosen

SUBSET_TRAIN = build_subset(TRAIN_ROWS)
SUBSET_VALIDATION = build_subset(VALIDATION_ROWS)
SUBSET_TRAIN_IDS = {r["document_id"] for r in SUBSET_TRAIN}
SUBSET_VALIDATION = [r for r in SUBSET_VALIDATION
                     if r["document_id"] not in SUBSET_TRAIN_IDS]  # disjoint

SUBSET_TRAIN_CONTRACTS = [build_contract(r) for r in SUBSET_TRAIN]
SUBSET_VALIDATION_CONTRACTS = [build_contract(r) for r in SUBSET_VALIDATION]
SUBSET_TYPES = sorted({e.entity_type for r in SUBSET_VALIDATION for e in r["entities"]})
SUBSET_REQUIRED = required_supervised_types(SUBSET_TYPES)
SUBSET_IDS_HASH = hashlib.sha256(
    json.dumps({"train": sorted(SUBSET_TRAIN_IDS),
                "validation": sorted(r["document_id"] for r in SUBSET_VALIDATION)},
               sort_keys=True).encode()).hexdigest()
print(json.dumps({
    "stage": "stage3_subset",
    "recipe": SELECTED_RECIPE.name,
    "train_examples": len(SUBSET_TRAIN),
    "validation_examples": len(SUBSET_VALIDATION),
    "sources": sorted({r["source_dataset"] for r in SUBSET_TRAIN}),
    "types_present_in_validation": SUBSET_TYPES,
    "required_types": list(SUBSET_REQUIRED),
    "subset_ids_sha256": SUBSET_IDS_HASH,
    "disjoint": True,
}, indent=2, sort_keys=True))

outcome = train_recipe(
    SUBSET_TRAIN, SUBSET_TRAIN_CONTRACTS, SELECTED_RECIPE, epochs=4,
    accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_rows=SUBSET_VALIDATION, eval_contracts=SUBSET_VALIDATION_CONTRACTS,
    history_path=SUBSET_DIR / "history.jsonl")
metrics = outcome["best_metrics"]
print(json.dumps({"stage": "subset_best_vs_final",
                  **outcome["best_final"].as_dict()}, indent=2, sort_keys=True))

best_path = SUBSET_DIR / "best.pt"
payload = e4_checkpoint_payload(
    mode="subset", config_sha256=CODE_SHA256, model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_MODEL_REVISION, parameter_count=0,
    recipe_name=SELECTED_RECIPE.name)
payload["model_state"] = outcome["best_state"]
torch.save(payload, best_path)
before_metrics, _bm, _bu = evaluate_state(
    outcome["best_state"], SUBSET_VALIDATION, SUBSET_VALIDATION_CONTRACTS)
reloaded = torch.load(best_path, map_location="cpu", weights_only=False)
reject_superseded_checkpoint(reloaded)
# Same real reproduction contract as Stage 2: the SAME best state, evaluated in
# a fresh model before serialization and again after reload.
after_metrics, missing, unexpected = evaluate_state(
    reloaded["model_state"], SUBSET_VALIDATION, SUBSET_VALIDATION_CONTRACTS)
SUBSET_REPRODUCTION = ReproductionCheck(
    checkpoint_sha256=sha256_file(best_path),
    metrics_before=reproduction_metrics(before_metrics),
    metrics_after=reproduction_metrics(after_metrics),
    missing_keys=missing, unexpected_keys=unexpected,
    fresh_model_evaluated=True, examples_evaluated=len(SUBSET_VALIDATION),
    metrics_before_source=BEST_STATE_PRE_SERIALIZATION,
    metrics_after_source=BEST_STATE_POST_RELOAD,
    eval_mode=True, deterministic_evaluation=True)
print(json.dumps({"stage": "subset_save_reload", **SUBSET_REPRODUCTION.as_dict()},
                 indent=2, sort_keys=True))
reload_ok = assert_real_reproduction(SUBSET_REPRODUCTION)
del reloaded
for stale in SUBSET_DIR.glob("epoch_*.pt"):
    stale.unlink()   # only the best checkpoint is retained

SUBSET_RESULT = SubsetResult(
    recipe=SELECTED_RECIPE.name,
    validation_predicted_mentions=metrics["validation_predicted_total"],
    validation_gold_mentions=metrics["validation_gold_total"],
    validation_recall=metrics["validation_exact_recall"],
    validation_exact_f1=metrics["validation_exact_f1"],
    nnw_predictions=metrics["validation_nnw_predictions"],
    thw_predictions_by_type=metrics["thw_predictions_by_type"],
    gold_positive_background_rate=metrics["gold_positive_background_rate"],
    collapse_guard_fired=outcome["collapsed"],
    save_reload_reproduced=reload_ok, artifact_validator_ok=True)
passed, failures = SUBSET_RESULT.passes(required_types=SUBSET_REQUIRED)
detail = {**SUBSET_RESULT.as_dict(), "failures": list(failures),
          "subset_ids_sha256": SUBSET_IDS_HASH,
          "reproduction": SUBSET_REPRODUCTION.as_dict(),
          "schedule": outcome["schedule"].as_dict(),
          "stopped_reason": outcome["stopped_reason"],
          "order_composition": outcome["compositions"]}
print(json.dumps(detail, indent=2, sort_keys=True))

GateArtifact(
    stage="subset_smoke", passed=passed, recipe=SELECTED_RECIPE.name,
    config_sha256=CODE_SHA256, code_sha256=CODE_SHA256,
    corpus_sha256=CORPUS_SHA256, detail=detail,
).write(GATE_DIR / SUBSET_GATE_FILENAME)

if not passed:
    raise SystemExit(f"the subset gate failed: {failures}; full training is refused")
print(json.dumps({"stage": "stage3_passed", "recipe": SELECTED_RECIPE.name},
                 indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 4 - FULL T4 TRAINING
#
# assert_full_training_allowed re-reads BOTH gate artifacts, requires both to
# have passed, requires them to agree on the recipe, and re-checks the config,
# code and corpus hashes. Flipping RUN_STAGE4_FULL_TRAINING satisfies exactly
# one of those conditions.
# =============================================================================
RECIPE_NAME = assert_full_training_allowed(
    gate_dir=GATE_DIR, config_sha256=CODE_SHA256, code_sha256=CODE_SHA256,
    corpus_sha256=CORPUS_SHA256, confirmation=CONFIRM_STAGE4,
    enabled=RUN_STAGE4_FULL_TRAINING)
RECIPE = build_recipe(RECIPE_NAME)
print(json.dumps({"stage": "stage4_authorized", "recipe": RECIPE_NAME,
                  "initialization": "pinned_pretrained_base_fresh_head"},
                 indent=2, sort_keys=True))

TRAIN_CONTRACTS = [build_contract(r) for r in TRAIN_ROWS]
VALIDATION_CONTRACTS = [build_contract(r) for r in VALIDATION_ROWS]

HISTORY_PATH = CURRENT_DIR / "logs" / "training_history.jsonl"
HISTORY_PATH.write_text("", encoding="utf-8")
outcome = train_recipe(
    TRAIN_ROWS, TRAIN_CONTRACTS, RECIPE, epochs=FULL_EPOCHS,
    accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_rows=VALIDATION_ROWS, eval_contracts=VALIDATION_CONTRACTS,
    history_path=HISTORY_PATH)

STATUS = "COLLAPSED_NOT_TRAINED" if outcome["collapsed"] else "FULLY_TRAINED"
assert_not_collapsed_when_marking_trained(outcome["verdict"], STATUS)

resolved = build_e4_resolved_config(
    mode="full", recipe=RECIPE.as_dict(), optimizer=RECIPE.optimizer.as_dict(),
    schedule=RECIPE.schedule.as_dict(),
    order={"data_order": RECIPE.data_order, "compositions": outcome["compositions"]},
    accumulation=outcome["plan"].as_dict(), precision=PRECISION.as_dict(),
    weight_format=WEIGHT_FORMAT, model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_MODEL_REVISION, seed=SEED, max_words=MAX_WORDS,
    epochs=FULL_EPOCHS, early_stopping_patience=3)
(CURRENT_DIR / "resolved_config.json").write_text(
    json.dumps(resolved, indent=2, sort_keys=True) + "\n", encoding="utf-8")

# Local first, hash-verified, then Drive.
staged = RUNTIME_ROOT / "best.pt"
payload = e4_checkpoint_payload(
    mode="full", config_sha256=canonical_json_sha256(resolved),
    model_revision=PINNED_MODEL_REVISION, tokenizer_revision=PINNED_MODEL_REVISION,
    parameter_count=0, recipe_name=RECIPE.name)
payload["model_state"] = outcome["best_state"]
torch.save(payload, staged)
digest = sha256_file(staged)
reject_superseded_checkpoint(torch.load(staged, map_location="cpu", weights_only=False))
final = CURRENT_DIR / "checkpoints" / "best.pt"
final.write_bytes(staged.read_bytes())
if sha256_file(final) != digest:
    raise AssertionError("Drive copy digest does not match the verified local copy")

accounting = build_training_accounting(
    plan=outcome["plan"], precision=PRECISION, optimizer=RECIPE.optimizer,
    schedule=RECIPE.schedule, observed_optimizer_steps=outcome["steps"],
    observed_backward_passes=outcome["backwards"],
    observed_examples=outcome["examples"], recipe_name=RECIPE.name)
(CURRENT_DIR / "validation_metrics.json").write_text(
    json.dumps({**outcome["best_metrics"], **outcome["best_final"].as_dict(),
                **outcome["selector"].as_dict(),
                "status": STATUS, "collapse": outcome["verdict"].as_dict(),
                "training_accounting": accounting},
               indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({"stage": "full_training_complete", "status": STATUS,
                  "best_checkpoint_sha256": digest,
                  **outcome["selector"].as_dict()}, indent=2, sort_keys=True))


## Return-to-repository

Record in the next append-only audit:

* `gates/stage2_tiny_ablation.json` — the three-recipe comparison and the
  selected recipe, with the pass/fail reason for every candidate;
* `gates/stage3_subset_smoke.json` — the subset gate result and the realized
  per-epoch order composition;
* `validation_metrics.json` — final status, best epoch, and the collapse verdict;
* `logs/training_history.jsonl` — per-epoch total / positive / background loss.

Report the status exactly as recorded. `COLLAPSED_NOT_TRAINED` is a real outcome
and must never be presented as a trained model — that conflation is what Audits
0043 and 0044 exist to prevent.

No checkpoint, artifact or log from any stage belongs in Git.
